<a href="https://colab.research.google.com/github/carthomp99/ML-SARShipDetection-Tutorial/blob/main/ML-SARShipDetection-Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAR Ship Detection - YOLO and RT-DETR Comparison Tutorial
##### __by Carter Thompson__
##### updated: 4 May 2026
The goal of this tutorial is to show how to train, utilize, and compare the efficacy of two machine learning models for SAR Ship Detection. YOLO models have become mainstays in the digital image processing boom through the mid-2010s to the present; they are quick and easy to implement, but have sometimes lacked in accuracy (though newer models continue to improve on this). DETR transformer models are very robust and precise, but have lacked the speeds YOLO models could achieve - until now, with the development of RT-DETR models that operate in real time.

This tutorial will show you how to prepare, train, optimize, and test both of these models separately, and provide you tools to compare their performance metrics against one another if that interests you.

Section 1 explores dataset setup, providing resources on where to find helpful SAR image datasets and how to structure your directories for use down the line.

Section 2 is a more conceptual section, exploring the two models in use in greater detail. This includes information on the structure of the input, intermediate, and output layers produced by the models. This is very code-light, and may not be useful to a more seasoned ML user.

Section 3 explores optimization parameters that will be put to use in sections 4 and 5. This is another code-light section intended mostly for deepening understanding of ML concepts. In other words, I wrote this section out more for me than for the user, so that I could more fully understand the principles of machine learning at play here.

Section 4 shows how to train both models on SAR data. For each model there will be two parts: a design step, which relies on fivefold cross validation to investigate model performance, and a production step, which trains on the full dataset in order to produce a production model.

Section 5 utilizes the insights gained from section 3 as well as those learned from training the model. Fine-tuning the model is discussed in greater detail. A brief example of an optimization loop is provided with instruction on how to expand the scope to your needs.

Section 6 finally puts the models we have spent all this time developing to the test on novel data. I will demonstrate this on a subset of SSDD data that I have kept distinct up to this point, and provide instructions on how to use external data for a similar purpose.

Finally, in section 7, I will compare the results of the tests with the two models, and provide useful code to A) investigate performance of individual models and B) compare the models one against the other.

# References
Datasets, Github Repos, YT Vids, etc

[1] T. Zhang, X. Zhang, J. Li, X. Xu, B. Wang, X. Zhan, Y. Xu, X. Ke, T. Zeng, H. Su et al., “SAR ship detection dataset (SSDD): Official release and comprehensive data analysis,” Remote Sensing, vol. 13, no. 18, p. 3690, 2021, doi: 10.3390/rs13183690.

[2] G. Jocher and J. Qiu, Ultralytics YOLO26. 2026. [Online]. Available: https://github.com/ultralytics/ultralytics

[3] W. Lv et al., “DETRs Beat YOLOs on Real-time Object Detection.” 2023.

# Top Level Library Requirements
Provide a list of required libraries and how to install them. Make
sure to include appropriate versions of each library. Use `assert`
commands to ensure that the libraries versions are available.

Provide the installation code near the top of each tutorial, below
the scope. Give users the option to skip installations if running
the code for a second time. You can do this with a simple `if`
statement. You also may want to add `pip` commands for installing
different libraries.

In [1]:
import time
import datetime
import os
import random
import json
from collections import defaultdict
from pathlib import Path
import shutil
from google.colab import files

# Data Manipulation
import numpy as np
import pandas as pd

# Plotting/Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML Libraries
import torch
import torchvision
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold

# Suppress per-epoch text output
os.environ["YOLO_VERBOSE"] = "False"
os.environ["TQDM_DISABLE"] = "1"

# Ultralytics install and imports
try:
  from ultralytics import YOLO
except ImportError:
  %pip install -q ultralytics
  from ultralytics import YOLO
  from ultralytics import RTDETR
else:
  from ultralytics import RTDETR

# Dataset and Model Gets from GitHub
if not os.path.exists("SSDD.zip"):
  !wget https://github.com/carthomp99/ML-SARShipDetection-Tutorial/raw/main/SSDD.zip
  !unzip -q ./SSDD.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
--2026-05-04 20:36:35--  https://github.com/carthomp99/ML-SARShipDetection-Tutorial/raw/main/SSDD.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/carthomp99/ML-SARShipDetection-Tutorial/main/SSDD.zip [following]
--2026-05-04 20:36:36--  https://raw.githubusercontent.com/carthomp99/ML-SARShipDetection-Tutorial/main/SSDD.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61014353 (58M) [application/zip]
Saving to: ‘SSDD.zip’

SSDD.zip            100%[===================>]  58.19M   223MB/s    in 0.3s    

2026-05-04 

# Google Drive Mount
You can mount your Google Drive to this Colab runtime using [method].
Describe how to do this.


# Dataset Requirements
For the purposes of this tutorial I train, validate, and test my models on subsets of the SAR Ship Detection Dataset [1] using fivefold cross validation. A prepared version of this dataset specifically for this tutorial can be found on my GitHub: [github link] and was already imported above.

This dataset is composed of 1,160 SAR images, taken both at shore and offshore on open waters. The training set contains 928 images, and the testing set contains 232 images (80-20 split). The size on disk of the full dataset, including annotation information, is approximately 64.2 MB.

To access this dataset for use in this project or others, [tell them how to use wget correctly and refer up to the code snippet that will do this]. Downloads are also available [at the places they are available]

To use your own dataset, [tell how to change the wget to access a different set of data or how to upload files to the runtime]. Using [whatever the function names end up being] will allow you to prepare a test set with five equally-sized folds.

# Hardware
YOLO models are optimized to run well on both CPU and GPU hardware. If you are only interested in the YOLO portions of this tutorial, feel free to use the CPU runtime listed in Colab. For the sake of the RT-DETR portions and the comparisons between both models, this tutorial assumes usage of the free T4 GPU runtimes available through Colab. Ensure you have selected this option under `Runtime > Change runtime type` before attempting to run any of the code blocks listed below.

Note: If you ran the library steps before switching runtimes, ensure you run them again in the T4 GPU runtime or you will encounter errors later on.

# Approximate Execution Times

Approximate execution times were estimated based on the duration of each code block running in the Colab T4 GPU runtime. Results may vary.
- Library setup/installs: ~25s initial, < 1s for repeat runs
- Dataset setup: ~1s
- Load/Save YOLO:
- Load/Save DETR:
- Optimize YOLO:
- Optimize DETR:
- Test YOLO:
- Test DETR:
- Fine-Tune YOLO:
- Fine-Tune DETR:
- YOLO Cross Val: \~2hr 40min (mean \~32 min per fold)
- YOLO Production Model:
- DETR Cross Val:
- DETR Production Model:
- Comparison and Analysis:

# Expected Results (metrics)

# \#1. Dataset Tutorial

Provide sufficient information to define all elements of a dataset:

-   Explain how to setup a dataset using Google Drive. Explain the
    directory structure and provide any zip, unzip commands.
-   Provide code to play audio, display image, or play video based
    on what you are doing.
-   Describe what the inputs are. Is it 1D/2D/3D NumPy array?
-   Describe what the output labels are: What are the categories?
-   Describe how to extend the dataset. Show how to download and add
    an audio, image, or a plain video to the dataset.
-   Explain how to setup the training, validation, and testing
    datasets.

## Annotation Structure
The SSDD comes with COCO-style .json annotations. Ultralytics expects the following structure for its models: a single .txt file per image with each line in the file taking the form

`<class_id> <x_center> <y_center> <width> <height>`

There is only a single class_id for this tutorial as we are __only__ concerned with identifying ships among any and all other image data.

The following code converts COCO to the required form. For this tutorial I will run it on both the test and train sets of the SSDD. To run on your own data, simply modify the directory paths in the second block below.

In [2]:
def convert_coco(json_path, images_dir, labels_dir):
  os.makedirs(labels_dir, exist_ok=True)

  # ===== LOAD JSON =====
  with open(json_path, "r") as f:
      data = json.load(f)

  # ===== MAP IMAGE ID → IMAGE INFO =====
  images = {img["id"]: img for img in data["images"]}

  # ===== GROUP ANNOTATIONS BY IMAGE =====
  annots_by_image = defaultdict(list)
  for ann in data["annotations"]:
      annots_by_image[ann["image_id"]].append(ann)

  # ===== CONVERT =====
  missing_images = 0
  written_files = 0

  for image_id, img_info in images.items():
      file_name = img_info["file_name"]
      w, h = img_info["width"], img_info["height"]

      # IMPORTANT: adjust path if needed
      img_path = os.path.join(images_dir, file_name)

      if not os.path.exists(img_path):
          missing_images += 1
          continue

      label_path = os.path.join(
          labels_dir,
          file_name.replace(".jpg", ".txt")
      )

      lines = []

      for ann in annots_by_image.get(image_id, []):
          x, y, bw, bh = ann["bbox"]

          # COCO → YOLO normalization
          x_center = (x + bw / 2) / w
          y_center = (y + bh / 2) / h
          bw_norm = bw / w
          bh_norm = bh / h

          cls = ann["category_id"]

          lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}")

      # write only if annotations exist
      if lines:
          with open(label_path, "w") as f:
              f.write("\n".join(lines))
          written_files += 1

  # ===== SUMMARY =====
  print("Done.")
  print(f"Labels written: {written_files}")
  print(f"Missing images skipped: {missing_images}")

In [3]:
# Test set conversion
json_path = "./SSDD/annotations/test.json"
images_dir = "./SSDD/images/test"
labels_dir = "./SSDD/labels/test"

convert_coco(json_path, images_dir, labels_dir)

# Train set conversion
json_path = "./SSDD/annotations/train.json"
images_dir = "./SSDD/images/train"
labels_dir = "./SSDD/labels/train"

convert_coco(json_path, images_dir, labels_dir)

Done.
Labels written: 232
Missing images skipped: 0
Done.
Labels written: 928
Missing images skipped: 0


## Dataset Assembly

With labels generated appropriately, we now need to structure the training set to perform five-fold cross validation. To do this simply, we can utilize symlinks (symbolic links) which will act as shortcuts to our SSDD dataset files without directly copying each image 5 times.

The following code splits the training set, both images and labels, of the SSDD directory into five folds for training. It first creates an array of all file paths, then shuffles that array and divides it into five equal pieces. The `dataset` directory contains sub-directories for each of the five folds; each individual fold uses the contents of four of the five folds as training data and the fifth for validation. During the training process, we will test and validate across all five folds, and take the average performance.

You can use this fold-splitter for your own dataset by simply changing the image_dir and label_dir filepaths to your directory.

__NOTE: Deleting files will cause issues with the symlinks as they require the original file to reference back to. Please avoid deleting files or directories, and note that resetting the runtime will similarly dispose of the symbolic directory until the splitter is run again.__

If you wish to __permanently__ modify your dataset to this format, simply change the symlink code and instead copy.

In [9]:
image_dir = "SSDD/images/train"
label_dir = "SSDD/labels/train"

# Make an array of every image in the directory
images = [f for f in os.listdir(image_dir) if f.endswith(".jpg")]

# SKLearn helps us implement 5Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    fold_dir = f"dataset/fold_{fold}"

    for split, indices in [("train", train_idx), ("val", val_idx)]:
        os.makedirs(f"{fold_dir}/images/{split}", exist_ok=True)
        os.makedirs(f"{fold_dir}/labels/{split}", exist_ok=True)

        for i in indices:
            img = images[i]
            lbl = img.replace(".jpg", ".txt")

            # Define target paths for symlinks
            target_image_path = f"{fold_dir}/images/{split}/{img}"
            target_label_path = f"{fold_dir}/labels/{split}/{lbl}"

            # Remove existing symlinks if they exist to prevent FileExistsError
            if os.path.exists(target_image_path):
                os.unlink(target_image_path)
            if os.path.exists(target_label_path):
                os.unlink(target_label_path)

            # symlink (fast & space-efficient) or copy
            os.symlink(
                os.path.abspath(f"{image_dir}/{img}"),
                target_image_path
            )
            os.symlink(
                os.path.abspath(f"{label_dir}/{lbl}"),
                target_label_path
            )
        # YAML generator per-fold
        yaml_content = f"""
path: {fold_dir}
train: images/train
val: images/val

names:
  0: ship
""".strip()

        yaml_path = os.path.join(fold_dir, f"data_fold_{fold}.yaml")

        with open(yaml_path, "w") as f:
            f.write(yaml_content)

print(f"Split complete at {datetime.datetime.now().strftime("%c")}.")

Split complete at Mon May  4 20:58:57 2026.


The following code creates .yaml files necessary for the creation of a population model using __100%__ of the dataset, rather than cross validating on a subset of data.

In [10]:
yaml_content = """
path: SSDD/images
train: ./train
val: ./test

names:
  0: ship
""".strip()

with open('SSDD/images/train.yaml', "w") as f:
            f.write(yaml_content)

# \#2. Model Description
This section explores the two models at work. You will see code examples of how to load and save models, as well as descriptions of the inner workings of each model. If you are already familiar with ML models you may wish to skip forward to the training, testing, and tuning steps further in this tutorial.

## \#2a. YOLO Model

### Loading
To load a pretrained YOLO model, ensure first that the YOLO class is properly imported from ultralytics libraries, then use the following code:

In [ ]:
from ultralytics import YOLO

# Load an existing YOLO model in for use, then display its information
model = YOLO("yolo26n.pt")
model.info()

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs


(260, 2572280, 0, 6.1192448)

You can replace `"yolo26n.pt" with any PyTorch (.pt extension) YOLO model, including [my-model-name.pt] provided at [the github link where my trained model will sit] to load in your own models for testing or other modification.

### Saving
You can export any Ultralytics model using the .export() method.. The default export format is a PyTorch file, but export() can also generate ONNX, TensorRT, and other common model formats. Further information on model exporting can be found in the Ultralytics documentation here [<--- PUT THE LINK].

In [ ]:
# Export trained model to default PyTorch format
model.export()

### Input Layer
YOLO input is an image tensor. YOLO models resize images to a fixed resolution for the sake of CNN efficiency. In cases where the resolutions do not already match ahead of time, the original aspect ratio can be preserved using padding. Images are also normalized for stability.

Every input is broken up into a grid, with each grid cell accounting for objects within itself. This will be explored in greater detail when discussing intermediate and outer layers.



### Output Layer
YOLO divides the image into an NxN grid. At each grid cell, the model predicts some number A anchors (this means that it suspects, at some level, that A objects exist in this cell).

Each anchor outputs a vector containing the predicted center coordinates of the predicted object, the width and height of the bounding box around the object, the "objectness" (likelihood that the thing being detected is actually an object), and the class it predicts this object is. Modern YOLO models use sigmoid activation functions for both objectness and class probabilities.

YOLO learns by minimizing bounding box loss, objectness loss, and classification loss. Bounding box losses are measured with intersection over union (IoU) functions, which measure overlap with the ground truth. Less overlap means high loss. Objectness losses are measured using binary cross-entropy (BCE) functions. Because "objectness" is a boolean measurement - true if there is an object, false if not - BCE can identify and severely punish differences in the predicted objectness and ground truth objectness. Classification losses typically use BCE functions as well.

### Intermediate Layers
YOLO is composed of three major parts. The CNN backbone of YOLO models extracts and maps features at different scales/resolutions for further analysis. Deeper layers have smaller resolutions, meaning they have very low detail but high semantics; shallow layers use higher resolutions but lack the context to describe objects within them.

The "neck" of the YOLO model combines these feature maps to maximize information. It utilizes a feature pyramid network (FPN) to send the semantic meaning from deep layers to shallow layers. Essentially, it uses a small resolution map (e.g. 20x20), upsamples it, and applies it to the medium resolution map (e.g. 40x40). Then it repeats this by upsampling that output and applying it to the large resolution (e.g. 80x80) map. This helps improve small-object detection.

The reverse of this is also done using a path aggregation network (PAN). This takes the FPN outputs and downsamples them back to their low resolutions, improving localization data for large objects that the FPN may have missed. These two processes create a feedback loop that helps identify objects and classifications at any scale.

Finally, now that we have rich feature maps at every scale, we need to convert them into actual bounding boxes and class predictions. The "head" of this model actually varies depending on the resolution of the map it receives. Detection heads include a small stack of convolutional layers that end with a prediction layer. This is what generates the actual anchors per each grid cell.

## \#2b. RT-DETR Model

Ultralytics provides pretrained RT-DETR models within their Python API. These models can then be trained and subsequently saved for further use.

### Loading
To load a pretrained RT-DETR model, ensure first that the RTDETR class is properly imported from ultralytics libraries, then use the following code:

In [ ]:
from ultralytics import RTDETR

# Load an existing RT-DETR model in for use, then display its information
model = RTDETR("rtdetr-l.pt")
model.info()

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs


(449, 32970476, 0, 108.3437056)

You can replace `"rtdetr-l.pt" with any PyTorch (.pt extension) RT-DETR model, including [my-model-name.pt] provided at [the github link where my trained model will sit] to load in your own models for testing or other modification.

### Saving
You can export any Ultralytics model using the .export() method.. The default export format is a PyTorch file, but export() can also generate ONNX, TensorRT, and other common model formats. Further information on model exporting can be found in the Ultralytics documentation here [<--- PUT THE LINK].

In [ ]:
# Export trained model to the default PyTorch format
model.export()

### Input Layer
RT-DTER input is an image tensor. Images are resized and normalized. Unlike the YOLO model, no anchor boxes are defined at the input stage.

### Output Layer
The model outputs a fixed number of object queries predicting bounding boxes and class probabilities. This is another difference from the YOLO model, which predicts per grid cell of the image. Each query produces a bounding box and class probabilities within that bounding box. This includes a "no object class" whose probability is high when nothing is detected. Query bounding boxes use sigmoid activation functions, while the class probabilities use softmax.

Like other DETR models, RT-DETR uses bipartite matching which uses a cost function to find the best one-to-one match between the predicted boxes and the ground truth. Both classification costs and bounding box costs are measured based on the probabilities calculated by the activation functions. The bounding box cost specifically is composed of both an L1 distance component and an intersection over union (IoU) component.

Many more queries than objects are generated per image, but the unmatched predictions are not useless - they are treated as "no object" objects with only classification losses applied. Thus, the model learns from its empty space as well as its matches. This feature of DETR models eliminates duplicate detections of objects, therefore removing the overhead costs of non-max suppression entirely.

### Intermediate Layers
RT-DETR is a hybrid model that uses both a CNN and a transformer. The backbone is a CNN feature extractor that converts the raw image into a feature map. Early layers of the map may only focus on edges of objects or textures. By the middle stages, specific shapes start to appear, and deeper and deeper layers have more complex and complete objects mapped out.

These feature maps are flattened into feature vectors, which are subsequently encoded with position data. From there, each "location" can look at and compare itself with each other location via self-attention. Patterns and relationships emerge from this self-attention that is subsequently decoded.

Actual object detection is performed by a transformer decoder. Some number of learnable vectors (queries) are initialized, and each begins to interact with the encoded features. Queries learn from each other just as feature vectors did through self-attention, and they grow more and more "attached" to features through training epochs until the final output layer is generated and objects and queries are matched.

# \#3. Model Optimization

This section describes how to optimize/fine-tune these two models. I will discuss the optimization algorithms, learning rates, batch sizes, epoch counts, and training vs validation graphs with helpful explanations to assist in the optimization process. I will explore the values I picked in training my own model and why. Understanding these parameters and how to modify them will help you tune your own model more precisely to your needs.

I will demonstrate the full optimization loop in section 5 of this tutorial. The purpose of this section is to explore the "knobs" to be tuned and discuss default values I have selected to train my dataset.

The following terms will be helpful in understanding the model-specific content below.

## Optimization algorithm and parameters:

Both models utilize the AdamW optimizer. The relevant parameters are learning rate, weight decay, and betas.
- Learning rate - controls how big a step the model takes when updating weights after each batch. High learning rate means aggressive and fast corrections, but could also lead to instability and overcorrection. You may sometimes wish to implement a lower introductory LR to prevent early instability; similarly, you can introduce decay to tune more finely over time.
- Weight decay - prevents overfitting by keeping weights small unless absolutely necessary. Essentially, weight decay discourages the model from relying too heavily on any single feature. __This is especially pertinent in SAR applications, ensuring the model learns more general ship features rather than memorizing specific conditions ships appear in in the data.__
- Betas - Adam's algorithms use two moving averages, known as $β_1$ and $β_2$, to control/oppose loss. In any given epoch, the model knows its current state and its past movements. We use these betas to "oppose the flow of loss" as it were - the gradients.
   - $β_1$, the momentum, controls how much past gradients influence the current update. This controls direction smoothing. Essentially, each epoch has a "direction" that drives up loss the most. We want to be aware of that to minimize the loss, but as these directions can change with each step, we use a weighted average to not overcorrect or zigzag.
   - $β_2$, the variance, controls the stability of step size - how much the model adapts based on variations in the gradient. Between epochs, the "magnitude" of the gradient can also change drastically - sometimes there are very dramatic swings. If $β_1$ is a mean, $β_2$ should naturally be a variance. If past gradients have been very large, we want to take more careful, controlled steps until we stabilize.

## Batch size:
We compute our gradients on batches, not on the entire set. This is especially true of a project like this with a relatively small dataset. Many factors play into selection of batch size, including but not limited to:
- Generalization - small batches tend to improve generalization because they regularize over time. Essentially, a smaller batch is less likely to have outliers meaning that over repeated epochs the calculations will smooth out more consistently.
- Memory usage - this increases with batch size, as one might expect.
- Gradient stability - We don't just want to observe the loss, we want to learn and minimize it over time. Batch size affects gradient noise which has a cascading effect into future epochs. We want to be somewhat stable so that we're moving steadily in a helpful direction, but we also want to be able to escape when outliers are encountered. Note that the Betas also affect this.

## Number of epochs:
How many times does the model repeat its learning processes? This varies between models, as each has their own convergence rates. Early stopping is a mechanism that stops the training process if no improvement is detected after a certain number of epochs have passed. Operating with little improvement can actually be detrimental, especially in SAR ship detection contexts, as it allows the model time to micro-adjust to __specific conditions in the data,__ such as specific background noise patterns, which will harm its usefulness on future, unseen data.

## Train vs val graphs:
The general goal is for training losses to decrease steadily, with validation decreasing to a plateau. The more similar these curves are, the better (generally). Certain trends in the curves can indicate important discrepancies:
- Validation loss being higher than training loss can suggest overfitting. Consider early stopping or stronger augmentation to minimize noise effects
- Both losses remaining high can indicate underfitting (too little learning at each step). Consider increasing either learning rate or number of epochs
- Fluctuations in validation can occur as a result of the noisy radar signatures of SAR images. Consider using larger batch sizes if possible, or smoothing the LR scheduling.

## \#3a. YOLO Optimization
Modern YOLO models use an AdamW optimizer. The relevant parameters for optimization are the LR, weight decay, and betas, as discussed above. When training my YOLO model, I used a starting learning rate of 1e-3, implementing cosine decay using built-in Ultralytics methods. I also used a 5-epoch warmup LR. I used a weight decay of 5e-4, a good baseline for SAR applications, and left the betas at their default values ($β_1$ = 0.937, $β_2$ = 0.999). __I *highly* recommend you leave the betas the same unless you need very very fine tuning and you know what you are doing.__

I used a 16 image batch size. For a custom dataset with high-resolution SAR images, it may behoove you to decrease this considerably. Conversely, if you have access to premium features of Google Colab, you may wish to utilize a stronger GPU runtime and *increase* the batch size.

I trained in 150 epochs with early stopping patience of 20. [UPDATE ANY AND ALL OF THIS IF YOU END UP CHANGING IT!]

## \#3b. RT-DETR Optimization
RT-DETR is a much more sensitive model than YOLO26; I adjusted my starting parameters to account for this. I used a starting LR of 1e-4, same for the weight decay, with default betas (0.937, 0.999).

I used an 8 image batch size to account for the heavier processing-per-epoch of RT-DETR. I once again trained in 150 epochs with early stopping patience of 20.

# \#4. Full Training
Run the loop multiple times and show that the validation and testing
losses converge to a good number. If the losses do not converge, you
will need to add data augmentation.

## \#4a. Top-to-bottom YOLO Training

### Fivefold CV

In [ ]:
if (torch.cuda.is_available()):
  print(f"Starting to train and validate YOLO on a {torch.cuda.get_device_name(0)}")
  print(f"Official start time: {datetime.datetime.now().strftime("%c")}. Godspeed\n")

  # Iterate across all folds
  fold_results = []

  for fold in range(5):
      model = YOLO("yolo26n.pt")

      run_name = f"fold_{fold}"

      results = model.train(
          data=f"dataset/fold_{fold}/data_fold_{fold}.yaml",
          epochs=150,
          batch=16,
          device=0,

          optimizer="AdamW",
          lr0=1e-3,
          lrf=0.01,
          cos_lr=True,
          warmup_epochs=5,
          weight_decay=5e-4,

          patience=20,
          plots=True,
          project="yolo_runs",
          name=run_name,
          exist_ok=True,

          verbose=False
      )

      fold_results.append(f"yolo_runs/{run_name}/results.csv")
      print(f"Finished fold {fold} at {datetime.datetime.now().strftime("%c")}")
else:
  print("Starting to train and validate YOLO on CPU")
  print(f"Official start time: {datetime.datetime.now().strftime("%c")}. Godspeed\n")

  # Iterate across all folds
  fold_results = []

  for fold in range(5):
      model = YOLO("yolo26n.pt")

      run_name = f"fold_{fold}"

      results = model.train(
          data=f"dataset/fold_{fold}/data_fold_{fold}.yaml",
          epochs=150,
          batch=16,
          device='cpu',

          optimizer="AdamW",
          lr0=1e-3,
          lrf=0.01,
          cos_lr=True,
          warmup_epochs=5,
          weight_decay=5e-4,

          patience=20,
          plots=True,
          project="yolo_runs",
          name=run_name,
          exist_ok=True,

          verbose=False
      )

      fold_results.append(f"yolo_runs/{run_name}/results.csv")
      print(f"Finished fold {fold} at {datetime.datetime.now().strftime("%c")}")

### CV Analysis
We now have five folds worth of insights on the training process, including confusion matrices, F1-confidence curves, and train vs val loss curves. The following code will process the output data cleanly and digestibly.

This helper function loads the results.csv file from each fold directory:

In [9]:
def load_results_csv(folder):
    path = Path(folder) / "results.csv"
    df = pd.read_csv(path)

    # clean column names
    df.columns = df.columns.str.strip()

    return df

This helper computes the F1 score manually to overlay the scores-per-fold:

In [10]:
def compute_f1(df):
    precision = df["metrics/precision(B)"]
    recall = df["metrics/recall(B)"]
    return 2 * (precision * recall) / (precision + recall + 1e-16)

This function actually plots and displays the F1 curves:

In [11]:
def plot_f1_curves(fold_dirs):
    plt.figure(figsize=(10, 6))

    for i, f in enumerate(fold_dirs):
        df = load_results_csv(f)
        f1 = compute_f1(df)

        # confidence proxy: Ultralytics doesn't always store per-confidence sweep
        # so we approximate using epochs as x-axis
        plt.plot(f1, label=f"Fold {i}")

    plt.title("F1 Curve Across 5 Folds")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

This method aggregates the curves for the train vs val loss curves to average in epoch:

In [14]:
def aggregate_curves(fold_dirs, column):
    curves = []

    for f in fold_dirs:
        df = load_results_csv(f)

        if column not in df.columns:
            raise ValueError(f"{column} not found in {f}")

        curves.append(df[column].values)

    # pad to same length
    max_len = max(len(c) for c in curves)
    padded = []

    for c in curves:
        if len(c) < max_len:
            c = np.pad(c, (0, max_len - len(c)), mode="edge")
        padded.append(c)

    return np.mean(padded, axis=0)

And finally, this last method plots the average train vs val loss curve for all five folds.

In [15]:
def plot_avg_train_val(fold_dirs):
    metrics = {
        "train/box_loss": "Train Box Loss",
        "train/cls_loss": "Train Cls Loss",
        "val/box_loss": "Val Box Loss",
        "metrics/mAP50(B)": "mAP50",
        "metrics/mAP50-95(B)": "mAP50-95"
    }

    plt.figure(figsize=(12, 8))

    for col, label in metrics.items():
        try:
            avg = aggregate_curves(fold_dirs, col)
            plt.plot(avg, label=label)
        except Exception as e:
            print(f"Skipping {col}: {e}")

    plt.title("Averaged Train vs Val Curves (5-Fold CV)")
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

Running this code uses all these helper methods to plot data nicely.

In [17]:
fold_dirs = [
    "runs/detect/yolo_runs/fold_0",
    "runs/detect/yolo_runs/fold_1",
    "runs/detect/yolo_runs/fold_2",
    "runs/detect/yolo_runs/fold_3",
    "runs/detect/yolo_runs/fold_4",
]

plot_f1_curves(fold_dirs)
plot_avg_train_val(fold_dirs)

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/yolo_runs/fold_0/results.csv'

<Figure size 1000x600 with 0 Axes>

This fivefold cross validation process yields important insights into model performance, allowing access to relevant statistics and metric information before evaluating on the full data. We can use those insights to develop a full model ready for production.

In the *real world,* this would be the part of the process where you notice how much better the RT-DETR model works, and you switch to that. But as I wish to be thorough, I will provide code to produce production models in both formats even if YOLO does a worse job at this application.

To do this, we simply train once more on a YOLO26n.pt model, this time accessing *all* the data.

### Production Model


In [ ]:
if (torch.cuda.is_available()):
  print(f"Starting to train and validate YOLO on a {torch.cuda.get_device_name(0)}")
  print(f"Official start time: {datetime.datetime.now().strftime("%c")}. Godspeed\n")
  model = YOLO("yolo26n.pt")
  results = model.train(
      data=f"SSDD/images/train.yaml",
      epochs=120,
      batch=16,
      device=0,

      optimizer="AdamW",
      lr0=1e-3,
      lrf=0.01,
      cos_lr=True,
      warmup_epochs=5,
      weight_decay=5e-4,

      patience=20,
      plots=True,
      project="yolo_production",
      name="yolo_production",
      exist_ok=True,

      verbose=False
  )
else:
  print(f"Starting to train and validate YOLO on a CPU")
  print(f"Official start time: {datetime.datetime.now().strftime("%c")}. Godspeed\n")
  model = YOLO("yolo26n.pt")
  results = model.train(
      data=f"SSDD/images/train.yaml",
      epochs=120,
      batch=16,
      device='cpu',

      optimizer="AdamW",
      lr0=1e-3,
      lrf=0.01,
      cos_lr=True,
      warmup_epochs=5,
      weight_decay=5e-4,

      patience=20,
      plots=True,
      project="yolo_production",
      name="yolo_production",
      exist_ok=True,

      verbose=False
  )
  print(f"Finished training at {datetime.datetime.now().strftime("%c")}")

Starting to train and validate RT-DETR on a CPU
Official start time: Mon May  4 21:06:23 2026. Godspeed



In [ ]:
# Save the entire run
shutil.make_archive(
    "yolo_production_backup",
    "zip",
    "runs/detect/yolo_production/yolo_production"
)
files.download("yolo_production_backup.zip")

## \#4b. Top-to-bottom DETR Training

In [ ]:
if (torch.cuda.is_available()):
  print(f"Starting to train and validate RT-DETR on a {torch.cuda.get_device_name(0)}")
  print(f"Official start time: {datetime.datetime.now().strftime("%c")}. Godspeed\n")
  # Iterate across all folds
  fold_results = []

  for fold in range(5):
      model = RTDETR("rtdetr-l.pt")

      run_name = f"fold_{fold}"

      results = model.train(
          data=f"dataset/fold_{fold}/data_fold_{fold}.yaml",
          epochs=150,
          batch=8,
          device=0,

          optimizer="AdamW",
          lr0=1e-4,
          lrf=0.01,
          cos_lr=True,
          warmup_epochs=5,
          weight_decay=1e-4,

          patience=20,
          plots=True,
          project="detr_runs",
          name=run_name,
          exist_ok=True,

          verbose=False
      )

      fold_results.append(f"detr_runs/{run_name}/results.csv")
      print(f"Finished fold {fold} at {datetime.datetime.now().strftime("%c")}")

# \#5. Basic Fine-Tuning
Download a pre-trained model, setup the optimization loop, and show
that the loss function is getting reduced. Retrain the pre-trained
model for a few iterations to demonstrate the optimization process.

## \#5a. Fine-Tuning the YOLO Model

## \#5b. Fine-Tuning the RT-DETR Model

# \#6. Basic Testing Tutorial
Load the pre-trained model using wget to download it and run it on one test example.

## \#6a. Testing the YOLO Model

## \#6b. Testing the DETR Model

# \#7. Comparison and Analysis Tutorial
Where I will give a tutorial on how to take the outputs and meaningfully compare them to understand the differences/improvements